# Kaggriculture Orbit-style GPU training

This notebook clones the Kaggriculture training code from GitHub, uses one Colab GPU for compact-policy updates, and uses two CPU workers for isolated simulator rollouts. Checkpoints, trajectories, and exported artifacts are stored on Google Drive so the run can resume after a disconnect.

Staged PPO workflow: run with 16, evaluate, then change the target to 32 and rerun the training cell. Candidate artifacts are retained separately, so each stage can be evaluated without overwriting the existing current artifact.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess

repo_url = 'https://github.com/dzlab/kaggle.git'
repo_branch = 'feat/kaggriculture-agent'
clone_root = Path('/content/kaggle')
project_root = clone_root / 'Kaggriculture'
drive_root = Path('/content/drive/MyDrive')
if not drive_root.exists():
    drive.mount('/content/drive')
else:
    print(f'Drive already mounted at {drive_root}')
os.chdir('/content')
if clone_root.exists():
    shutil.rmtree(clone_root)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', repo_branch,
    '--single-branch', repo_url, str(clone_root),
], check=True)
print(f'Cloned {repo_url} ({repo_branch}) into {project_root}')

In [ ]:
%cd /content/kaggle/Kaggriculture
!pip -q install -e '.[training]'
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime in Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
from pathlib import Path

run_dir = Path('/content/drive/MyDrive/kagriculture-orbit')
run_dir.mkdir(parents=True, exist_ok=True)
trajectory_path = run_dir / 'bootstrap-trajectories.jsonl'

# CPU-bound simulator collection; the model update below runs on CUDA.
!python scripts/collect_trajectories.py --seeds 8 --start-seed 0 --steps 96 --opponents pass random starter --seats 0 1 --workers 2 --output {trajectory_path}

In [ ]:
import importlib
import scripts.train_policy as train_policy
importlib.reload(train_policy)
import scripts.colab_train as colab_train
importlib.reload(colab_train)
from scripts.export_policy import export_checkpoint
from scripts.train_policy import make_fresh_rollout_fn, train_behavior_clone

# Change this one visible target for the next bounded PPO stage.
ppo_target_steps = 16
candidate_tag = f"ppo{ppo_target_steps}"
stage_checkpoint_path = run_dir / f"orbit-policy-{candidate_tag}.pt"
stage_artifact_path = run_dir / f"orbit-policy-{candidate_tag}.json"
current_checkpoint_path = run_dir / 'orbit-policy.pt'

training_steps = 25
training_batch_size = 256
training_seed = 7
training_device = 'cuda'
training_checkpoint_interval = 25
training_prior_checkpoint = None
training_offline_ppo_fallback = False
training_contract = train_policy.build_training_contract(
    input_path=trajectory_path,
    steps=training_steps,
    batch_size=training_batch_size,
    seed=training_seed,
    ppo_steps=ppo_target_steps,
    device=training_device,
    checkpoint_interval=training_checkpoint_interval,
    prior_checkpoint=training_prior_checkpoint,
    offline_ppo_fallback=training_offline_ppo_fallback,
)

resume_selection = colab_train.select_resume_checkpoint(
    (current_checkpoint_path, *sorted(run_dir.glob('orbit-policy-ppo*.pt'))),
    training_contract=training_contract,
)
if resume_selection is None:
    saved_ppo_target = None
    resume_checkpoint = None
else:
    saved_ppo_target = resume_selection.ppo_target_steps
    resume_checkpoint = resume_selection.path
allow_ppo_extension = saved_ppo_target is not None and saved_ppo_target < ppo_target_steps
print('Stage:', candidate_tag, 'target PPO steps:', ppo_target_steps)
print('Resume checkpoint:', resume_checkpoint or 'none; starting a new run')
print('allow_ppo_extension:', allow_ppo_extension)

def export_current(*, output_path, **_kwargs):
    export_checkpoint(stage_checkpoint_path, output_path)
    return output_path

fresh_rollout = make_fresh_rollout_fn(
    run_directory=run_dir / 'ppo-rollouts',
    candidate_artifact_callback=export_current,
    seeds=(0, 1, 2, 3),
    steps=97,
    workers=2,
)

metadata = train_behavior_clone(
    input_path=trajectory_path,
    output_path=stage_checkpoint_path,
    steps=training_steps,
    batch_size=training_batch_size,
    seed=training_seed,
    ppo_steps=ppo_target_steps,
    device=training_device,
    checkpoint_interval=training_checkpoint_interval,
    resume_checkpoint=resume_checkpoint,
    allow_ppo_extension=allow_ppo_extension,
    prior_checkpoint=training_prior_checkpoint,
    rollout_fn=fresh_rollout,
    offline_ppo_fallback=training_offline_ppo_fallback,
    candidate_artifact=stage_artifact_path,
)
export_checkpoint(stage_checkpoint_path, stage_artifact_path)
print(metadata)
print('checkpoint:', stage_checkpoint_path)
print('artifact:', stage_artifact_path)

In [ ]:
# Verify the exported dependency-free candidate in the local simulator.
import re
import subprocess
import sys

smoke = subprocess.run(
    [
        'python', 'scripts/run_local.py',
        '--opponent', 'pass',
        '--seed', '0',
        '--steps', '96',
        '--replay', str(run_dir / 'candidate-smoke.json'),
        '--candidate-artifact', str(stage_artifact_path),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(smoke.stdout, end='')
# kaggle-environments 1.32.7 probes an unavailable optional OpenSpiel game
# during startup. Remove only that known diagnostic; preserve all other stderr.
clean_stderr = re.sub(
    r"OpenSpiel exception: Unknown game 'python_ant_foraging'\. Available games are:\n.*?\nzerosum\n?",
    '',
    smoke.stderr,
    flags=re.DOTALL,
)
if clean_stderr:
    print(clean_stderr, file=sys.stderr, end='')
print('Smoke test completed successfully. Re-run the training cell after a disconnect; it will resume from the latest compatible PPO stage checkpoint when present.')